---
title: "Lean: Theorem Prover"
author: "Vahram Poghosyan"
date: "2026-03-07"
categories: ["Theorem Provers"]
format:
  html:
    toc: true
    toc-depth: 5
    code-fold: true
jupyter: python3
include-after-body:
  text: |
    <script type="application/javascript" src="../../javascript/light-dark.js"></script>
---

# About

[Lean](https://leanprover-community.github.io/) is an open-source theorem prover and functional programming language developed by Leonardo de Moura. The project originated at Microsoft Research in 2013, with the goal of bridging the gap between interactive and automated theorem proving. Lean 2 followed in 2015, but the most transformative leap came with Lean 4, released in 2021 — a ground-up rewrite
that reimagined Lean not just as a proof assistant but as a general-purpose programming language.

History:
- 2013 — Leonardo de Moura begins developing Lean at Microsoft Research, Redmond.
- 2015 — Lean 2 is released with improved tactic support and a growing mathematical library.
- 2017 — Lean 3 launches, gaining traction in the mathematics community. The [mathlib](https://mathlib-initiative.org/about/) library project begins, eventually becoming one of the largest formalized mathematics libraries in the world.
- 2021 — Lean 4 is released, featuring a completely rewritten codebase (Lean 4 is largely written in itself). It introduces a new macro/elaboration system, hygienic macros, and a compiler that produces efficient C code.
- 2023–present — Lean continues to grow, with [mathlib](https://github.com/leanprover-community/mathlib4) fully ported to Lean 4 and adoption expanding in both academia and industry.

A Functional Programming Language

What sets Lean apart from many proof assistants is that it doubles as a full-fledged functional programming language. Lean 4 features a pure functional core based on dependent type theory, with support for: 
- Monadic IO
- Algebraic Data Types -- e.g. **sum types** like `Option` in Lean, or **product types** like `Point` which can be thought of as 2D coordinates with fields `x`, and `y` (for example)
- Pattern matching
- Type classes 
- Metaprogramming -- writing code that generates or manipulates other code (for example, Lean has **macros** which are syntax transformations, **extensions** for types, and **tactics** for proof automation)

Its compiler generates efficient native code via C, making it
practical for real-world software — not just formal verification. In this sense, Lean occupies a unique niche: you can write a mathematical proof and a performant program in the same language, using the same type system to guarantee correctness.

# Setup

1. Install Lean via [elan](https://github.com/leanprover/elan) (the Lean version manager):
   ```
   curl https://raw.githubusercontent.com/leanprover/elan/master/elan-init.sh -sSf | sh
   ```
2. Install the VS Code extension:
   1. Open Extensions (Cmd+Shift+X) → search "Lean 4" by **leanprover** → Install. 
   2. The extension provides the language server, diagnostics, and the interactive **Infoview** panel which is used to check proofs and see type information.
3. Create a new Lean project (**Lake** is the standard build system for Lean 4, replacing **leanpkg** from Lean 3):
   ```
   mkdir my-lean-project && cd my-lean-project
   lake init my_project
   lake build
   ```
4. Open the project in VS Code:

# Our First Proof

For our first proof, let's show that: 

::: {.callout-note title="Theorem" icon=false}
For any natural number `n`, `n + 0 = n`.
:::

This is a fundamental property of addition, and we can prove it using induction.

We're going to use the [Peano axioms](https://en.wikipedia.org/wiki/Peano_axioms) to define natural numbers and addition. Then, we'll use induction to prove the theorem. Here are the Peano axioms and the definition of addition.

1. `0` is a natural number.
   
The next four axioms describe the equality relation. Since they are logically valid in [first-order logic](https://web.stanford.edu/class/archive/cs/cs103/cs103.1232/lectures/04/Condensed%20Slides.pdf) with equality, they are not considered to be part of "the Peano axioms" in modern treatments. As a reminder, first-order logic augments [propositional logic](https://en.wikipedia.org/wiki/Propositional_logic) with quantifiers (like "for all" or $\forall$, and "there exists" or $\exists$), *functions* which map objects to other objects, and a way to talk about properties of objects using *predicates* (equality is a special predicate between 2 objects). The next 4 original Peano axioms are:

1. For every natural number `n`, `n = n`. That is, equality is reflexive.
2. For all natural numbers `n` and `m`, if `n = m`, then `m = n`. That is, equality is symmetric.
3. For all natural numbers `n`, `m` and `p`, if `n = m` and `m = p`, then `n = p`. That is, equality is transitive.
4. For all `n` and `m`, if `m` is a natural number and `n = m`, then `n` is also a natural number. That is, the natural numbers are closed under equality.

The remaining axioms define the arithmetical properties of the natural numbers. The naturals are assumed to be closed under a single-valued "successor" function `succ`.

5. If `n` is a natural number, then `succ n` (the successor of `n`) is also a natural number. That is, natural numbers are closed under the successor operation.
6. For all natural numbers `m` and `n`, if `succ m = succ n`, then `m = n`. That is, `succ` is an injection.
7. For every natural number n, S(n) = 0 is false. That is, there is no natural number whose successor is 0.

Here's how we can express these axioms and the proof in Lean:

```lean
-- Define the natural numbers (Peano axioms)
inductive Nat
| zero : Nat -- "Zero is a natural number"
| succ : Nat → Nat -- "If n is a natural number, then succ n is also a natural number"
-- The axiom of injectivity of 'succ' is automatically guaranteed by Lean's type theory. When we define an inductive type, Lean's kernel enforces that distinct constructor applications produce distinct values. So `succ m = succ n` can only hold if `m = n`.
-- The axiom that there is no natural number whose successor is zero is also automatically enforced by the structure of the inductive type. Since `zero` is a distinct constructor and `succ` always produces a new value, there is no way to construct a natural number `n` such that `succ n = zero`.

-- Define addition recursively ()
def add : Nat → Nat → Nat
| Nat.zero, m => m
| Nat.succ n, m => Nat.succ (add n m) -- i.e. `add(add(n,1), m) = add(1,add(n, m))` or, more colloquially, this is communativity of addition

-- Prove that n + 0 = n for all n
theorem add_zero (n : Nat) : add n Nat.zero = n := by
    induction n with
    | zero => simp [add]
    | succ d hd =>
      simp [add]
      rw [hd]
```

In this proof:
- We define the natural numbers using an *inductive type* `Nat`, where `zero` represents 0 and `succ` represents the successor function (e.g., `succ zero` is 1, `succ (succ zero)` is 2, etc.).
  - Specifically, `Nat` is defined as an inductive type with two constructors: `zero` and `succ`. The `zero` constructor represents the base case of 0, while the `succ` constructor takes a natural number and returns its successor. So the naturals are built up as: `Nat.zero`, `Nat.succ Nat.zero` (which is 1), `Nat.succ (Nat.succ Nat.zero)` (which is 2), and so on...
  
- We define addition recursively: adding zero to any number `m` gives `m`, and adding the successor of `n` to `m` gives the successor of `add n m`.
  - This is enough to define addition on the natural numbers because of recursion on the inductive type `Nat`. Since `Nat` has exactly two constructors (`zero` and `succ n`), pattern matching on those two cases is exhaustive — every possible natural number is either `Nat.zero` or `Nat.succ n` for some `n`. Lean verifies this at compile time.
  
- We then state the theorem:
  - `theorem add_zero (n : Nat) : add n Nat.zero = n` means "for any natural number `n`, adding zero to `n` gives back `n`". We can also write this closer to mathematical notation in Lean 4's syntax: `∀ n : Nat, add n Nat.zero = n`.
  - The proof uses induction on `n`. The base case is when `n` is `zero` (`with` introduces the case arms). The RHS of the base case, `simp [add]`, *simplifies* the goal (`add n Nat.zero = n`) using the definition of `add` as a [rewrite rule](https://lean-lang.org/doc/reference/latest/The-Simplifier/Rewrite-Rules/). It unfolds `add Nat.zero Nat.zero` to `Nat.zero` (per the definition) and closes the goal since `Nat.zero = Nat.zero` trivially.
  - The inductive case assumes the property holds for `d`, and proves it for `succ d` (the successor of `d`). 
    - The keyword `hd` is the *inductive hypothesis* (`add d Nat.zero = d`), which we state in the `proof` in the Lean 4 `induction` *tactic*.
    - First, we simplify the goal using `simp [add]`, which unfolds `add (succ d) Nat.zero` to `succ (add d Nat.zero)`.
    - Then, we use `rw [hd]` to rewrite the goal using the inductive hypothesis, replacing `add d Nat.zero` with `d`. This gives us `succ d = succ d`, which is trivially true, completing the proof.